# SkillsAgent — Colab runner with real TSFM (T4 GPU)

This notebook runs the **SkillsAgent ablation study** on a Colab GPU so `granite-tsfm` (forecasting + anomaly detection) executes for real instead of falling back to the mock.

**Before you start**

1. `Runtime → Change runtime type → GPU (T4)`
2. Put your project folder in Google Drive at e.g. `MyDrive/HPML/project/` containing `SkillsAgent/`, `AssetOpsBench/`, and `main.json`.
3. Run cells top-to-bottom.

**What this notebook does**

- Git-clones AssetOpsBench from GitHub + rsyncs SkillsAgent & WO sample CSVs from Drive (~1 min total)
- Installs `granite-tsfm` + SkillsAgent requirements
- Extracts per-asset CSVs from `main.json` once (→ `data/chillers/`)
- Writes a Colab `.env` (no CouchDB; uses `IOT_CSV_DIR` instead)
- Smoke-tests real TSFM on the T4
- Runs `eval_runner` on the representative subset and downloads results
- Optional **wandb** (Section 1b): set `WANDB_PROJECT` + Colab secret `WANDB_API_KEY` to log ablations to [wandb.ai](https://wandb.ai)

In [1]:
import os, subprocess

assert subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0, \
    "GPU not attached — Runtime → Change runtime type → T4 GPU"

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-8ce20d99-d77a-1ca3-fca7-912240faf2f4)



## 1. Fetch code to local disk

Drive → Colab FUSE is painfully slow for tens of thousands of small files (`.venv`, `.git` pack objects, cached artifacts), so we **git-clone AssetOpsBench from GitHub** and only rsync the small bits that aren't in the upstream repo:

- `AssetOpsBench/src/tmp/assetopsbench/sample_data/` — WO CSVs (~8 MB) needed for `USE_WO_SUBPROCESS`
- `SkillsAgent/` — your own code, with aggressive excludes

If you have **local edits** to AssetOpsBench, set `ASSETOPS_REPO` below to your fork, or revert to the old rsync path.

In [2]:
import os, subprocess, time
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT = "/content/drive/MyDrive/HPML_Project"
LOCAL_ROOT    = "/content/work"
ASSETOPS_REPO = "https://github.com/IBM/AssetOpsBench.git"
ASSETOPS_REF  = "main"

assert os.path.isdir(DRIVE_PROJECT), f"not found: {DRIVE_PROJECT}"
os.makedirs(LOCAL_ROOT, exist_ok=True)

aob_dir = f"{LOCAL_ROOT}/AssetOpsBench"

# 1a. AssetOpsBench: shallow git clone (seconds) instead of Drive rsync (hours).
#     If a prior attempt left a non-empty dir without `.git`, git clone exits 128
#     ("destination path already exists and is not an empty directory"); we wipe
#     partials and surface stderr so any real error is actually visible.
t0 = time.time()
if os.path.isdir(aob_dir) and not os.path.isdir(f"{aob_dir}/.git"):
    print(f"  wiping partial/empty {aob_dir}")
    subprocess.check_call(["rm", "-rf", aob_dir])

if not os.path.isdir(f"{aob_dir}/.git"):
    r = subprocess.run(
        ["git", "clone", "--depth=1", "--branch", ASSETOPS_REF,
         ASSETOPS_REPO, aob_dir],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print("--- git stdout ---\n" + (r.stdout or ""))
        print("--- git stderr ---\n" + (r.stderr or ""))
        raise RuntimeError(f"git clone failed (exit {r.returncode})")
    print(f"  AssetOpsBench cloned in {time.time()-t0:.1f}s")
else:
    print(f"  AssetOpsBench already at {aob_dir} — skipping clone")

# 1b. WO sample CSVs live at src/tmp/assetopsbench/sample_data/ — NOT in the
# upstream git repo, so rsync them (~8 MB) from Drive. Needed by FMSR
# (failure_codes, primary_failure_codes) and USE_WO_SUBPROCESS.
t0 = time.time()
drive_tmp = f"{DRIVE_PROJECT}/AssetOpsBench/src/tmp"
local_tmp = f"{aob_dir}/src/tmp"
if os.path.isdir(drive_tmp):
    os.makedirs(local_tmp, exist_ok=True)
    subprocess.check_call(["rsync", "-a", f"{drive_tmp}/", f"{local_tmp}/"])
    print(f"  WO sample_data rsynced in {time.time()-t0:.1f}s")
else:
    print(f"  SKIP src/tmp: not found at {drive_tmp} — WO tool will fall back to mock")

# 1c. SkillsAgent: small code-only rsync from Drive. Aggressive excludes skip
# OneDrive/MacOS junk and prior eval outputs (step 7 copies those back to Drive).
t0 = time.time()
subprocess.check_call([
    "rsync", "-a",
    "--exclude=.git", "--exclude=__pycache__", "--exclude=.pytest_cache",
    "--exclude=.venv", "--exclude=.DS_Store", "--exclude=eval_results",
    f"{DRIVE_PROJECT}/SkillsAgent/", f"{LOCAL_ROOT}/SkillsAgent/",
])
print(f"  SkillsAgent rsynced in {time.time()-t0:.1f}s")

SKILLS    = f"{LOCAL_ROOT}/SkillsAgent"
ASSETOPS  = f"{aob_dir}/src"
MAIN_JSON = f"{DRIVE_PROJECT}/main.json"
TSFM_REPORT = f"{DRIVE_PROJECT}/tsfm_report.csv"

print()
print("SKILLS   ", SKILLS)
print("ASSETOPS ", ASSETOPS)
print("TSFM_REPORT", TSFM_REPORT, "(exists)" if os.path.isfile(TSFM_REPORT) else "(MISSING)")
print("MAIN_JSON", MAIN_JSON, "(exists)" if os.path.isfile(MAIN_JSON) else "(MISSING)")

Mounted at /content/drive
  AssetOpsBench cloned in 2.8s
  WO sample_data rsynced in 25.7s
  SkillsAgent rsynced in 11.3s

SKILLS    /content/work/SkillsAgent
ASSETOPS  /content/work/AssetOpsBench/src
TSFM_REPORT /content/drive/MyDrive/HPML_Project/tsfm_report.csv (exists)
MAIN_JSON /content/drive/MyDrive/HPML_Project/main.json (exists)


## 2. Install dependencies

Two places deps need to land, because TSFM runs via `uv run` inside a separate venv:

1. **Colab system Python** — SkillsAgent's own deps (fast).
2. **`AssetOpsBench/.venv`** — `torch`, `transformers`, and `granite-tsfm` (the big one). Upstream `pyproject.toml` explicitly says *"tsfm_public must be installed separately"* and only lists `torch`/`transformers` in the optional `[tsfm]` group, so we do both.

First run: ~3–5 min (mostly pulling CUDA torch wheel). Cached on subsequent runs in the same session.

In [3]:
import subprocess, sys, time

# 2a. SkillsAgent's own Python deps → Colab system Python.
print("→ SkillsAgent requirements...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "-r", f"{SKILLS}/requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "ijson"])

# 2b. AssetOpsBench venv gets torch + transformers + granite-tsfm so the
#     `uv run python -c ...` TSFM subprocess can import tsfm_public.
print("→ uv (drives AssetOpsBench venv)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "uv"])

print("→ uv sync --group tsfm  (torch CUDA + transformers, first time is slow)...")
t0 = time.time()
subprocess.check_call(["uv", "sync", "--group", "tsfm"], cwd=aob_dir)
print(f"   uv sync done in {time.time()-t0:.1f}s")

venv_py = f"{aob_dir}/.venv/bin/python"
assert os.path.isfile(venv_py), f"venv python missing: {venv_py} — did uv sync succeed?"

print("→ granite-tsfm into AssetOpsBench/.venv (explicit --python target)...")
t0 = time.time()
# --python pins the install to THIS venv; otherwise uv's resolution can land
# the package somewhere else (cache-only, or Colab system) and tsfm_public
# stays unimportable from `uv run`.
subprocess.check_call([
    "uv", "pip", "install", "--python", venv_py,
    "granite-tsfm @ git+https://github.com/ibm-granite/granite-tsfm.git",
])
print(f"   granite-tsfm installed in {time.time()-t0:.1f}s")

# transformers 4.57 pins huggingface-hub <1.0, but resolution sometimes pulls
# hf-hub 1.x (Colab system env + granite-tsfm transitive). Force back to <1.0.
print("→ pinning huggingface-hub<1.0 (transformers 4.57 constraint)...")
subprocess.check_call([
    "uv", "pip", "install", "--python", venv_py,
    "huggingface-hub<1.0",
])

# Show that the four critical packages actually landed in this venv.
r = subprocess.run(
    ["uv", "pip", "list", "--python", venv_py],
    capture_output=True, text=True,
)
hits = [ln for ln in r.stdout.splitlines()
        if ln.lower().startswith(("granite-tsfm", "torch ", "transformers ",
                                  "huggingface-hub"))]
print("venv packages:\n  " + "\n  ".join(hits) if hits else "  (none matched)")

# Final sanity check via `uv run` — mirrors how tools.py calls TSFM.
probe = (
    "import torch, tsfm_public, sys; "
    "print('torch', torch.__version__, 'CUDA?', torch.cuda.is_available(), "
    "'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', "
    "'| tsfm_public OK @', tsfm_public.__file__)"
)
r = subprocess.run(["uv", "run", "--no-sync", "python", "-c", probe],
                   cwd=aob_dir, capture_output=True, text=True, timeout=180)
print("--- venv sanity:", r.stdout.strip())
if r.returncode != 0:
    print("--- stderr:", r.stderr.strip())
    raise RuntimeError("AssetOpsBench venv missing deps")

→ SkillsAgent requirements...
→ uv (drives AssetOpsBench venv)...
→ uv sync --group tsfm  (torch CUDA + transformers, first time is slow)...
   uv sync done in 36.5s
→ granite-tsfm into AssetOpsBench/.venv (explicit --python target)...
   granite-tsfm installed in 47.2s
→ pinning huggingface-hub<1.0 (transformers 4.57 constraint)...
venv packages:
  granite-tsfm                       0.3.6
  huggingface-hub                    0.36.2
  torch                              2.10.0
  transformers                       4.57.6
--- venv sanity: torch 2.10.0+cu128 CUDA? True device: NVIDIA A100-SXM4-40GB | tsfm_public OK @ /content/work/AssetOpsBench/.venv/lib/python3.12/site-packages/tsfm_public/__init__.py


## 3. Extract per-asset CSVs from `main.json`

One-shot: streams the 1.16 GB export and writes `chiller_6.csv` / `chiller_9.csv` (~1–2 min). `USE_IOT_SUBPROCESS` will be off, so `get_sensor_data` reads these.

In [4]:
CSV_DIR = f"{SKILLS}/data/chillers"
if not os.path.isfile(f"{CSV_DIR}/chiller_6.csv"):
    !python {SKILLS}/scripts/extract_main_json.py \
        --input {MAIN_JSON} --outdir {CSV_DIR} \
        --assets 'Chiller 6,Chiller 9' --max-rows 2000
else:
    print("CSVs already extracted — skipping")

!wc -l {CSV_DIR}/*.csv

CSVs already extracted — skipping
  1501 /content/work/SkillsAgent/data/chillers/chiller_6.csv
  1501 /content/work/SkillsAgent/data/chillers/chiller_9.csv
  3002 total


## 4. Write Colab `.env`

We skip CouchDB entirely. `IOT_CSV_DIR` drives `get_sensor_data`. `PATH_TO_MODELS_DIR` tells AssetOpsBench where the pretrained TTM checkpoints live.

**Paste your own API keys below** (watsonx is the default LLM provider).

In [5]:
from getpass import getpass


WATSONX_API_KEY    = os.environ.get("WATSONX_API_KEY")    or getpass("WATSONX_API_KEY: ")
WATSONX_PROJECT_ID = os.environ.get("WATSONX_PROJECT_ID") or input("WATSONX_PROJECT_ID: ")
GROQ_API_KEY       = os.environ.get("GROQ_API_KEY")       or ""
GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY")     or ""

env = f"""\
LLM_PROVIDER=watsonx
WATSONX_API_KEY={WATSONX_API_KEY}
WATSONX_PROJECT_ID={WATSONX_PROJECT_ID}
WATSONX_URL=https://us-south.ml.cloud.ibm.com
WATSONX_MODEL_ID=meta-llama/llama-4-maverick-17b-128e-instruct-fp8,meta-llama/llama-3-3-70b-instruct,mistral-large-2512
WATSONX_MAX_RETRIES=2
GROQ_API_KEY={GROQ_API_KEY}
GEMINI_API_KEY={GEMINI_API_KEY}
GEMINI_MODEL=gemini-2.5-flash

ASSETOPS={ASSETOPS}
PATH_TO_MODELS_DIR={ASSETOPS}/servers/tsfm/artifacts/tsfm_models
IOT_CSV_DIR={CSV_DIR}
COUCHDB_EXPORT_PATH={MAIN_JSON}

USE_IOT_SUBPROCESS=0
USE_TSFM_SUBPROCESS=1
USE_WO_SUBPROCESS=1
USE_FMSR_SUBPROCESS=1
WO_CSV_DIR={ASSETOPS}/tmp/assetopsbench/sample_data
TSFM_SUBPROCESS_TIMEOUT=600
TSFM_MIN_CONTEXT_ROWS=96
"""
open(f"{SKILLS}/.env", "w").write(env)
print("wrote", f"{SKILLS}/.env")
# Do not print full env — Colab/UI may log cell output and git may snapshot notebook outputs.
print(
    "Secrets: written to .env only (not echoed). "
    f"api_key_len={len(WATSONX_API_KEY)}, project_id_len={len(WATSONX_PROJECT_ID)}, "
    f"groq_len={len(GROQ_API_KEY)}, gemini_len={len(GEMINI_API_KEY)}"
)

wrote /content/work/SkillsAgent/.env
Secrets: written to .env only (not echoed). api_key_len=44, project_id_len=36, groq_len=0, gemini_len=0


## 4b. (Optional) Weights & Biases experiment tracking

If you set **`WANDB_PROJECT`**, `eval_runner` will log each task×condition row (cost, latency, `task_completion`, etc.) and finish the run when the ablation completes.

1. Create a [wandb](https://wandb.ai) account and copy your **API key**.
2. Run this cell after the previous cell (cell 4).
3. Additional WANDB environment variables will be append written to the .env file.

Optional env vars (set before `eval_runner`): **`WANDB_ENTITY`**, **`WANDB_RUN_GROUP`**, **`WANDB_RUN_NAME`**, comma-separated **`WANDB_TAGS`**. Set **`WANDB_LOG_ARTIFACT=1`** to upload `ablation_results.csv` at the end. **`WANDB_DISABLED=1`** turns logging off.

In [6]:
# --- Optional: Weights & Biases (same pattern as section 4: merge into `{SKILLS}/.env` + set os.environ) ---
import os
import subprocess
import sys
from getpass import getpass

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "wandb"])

WANDB_API_KEY = os.environ.get("WANDB_API_KEY") or getpass("WANDB_API_KEY (optional, Enter to skip): ").strip()

WANDB_PROJECT = (os.environ.get("WANDB_PROJECT") or "").strip() or "skillsagent-colab"
WANDB_ENTITY = (os.environ.get("WANDB_ENTITY") or "").strip()
WANDB_RUN_GROUP = (os.environ.get("WANDB_RUN_GROUP") or "").strip()
WANDB_RUN_NAME = (os.environ.get("WANDB_RUN_NAME") or "").strip()
WANDB_TAGS = (os.environ.get("WANDB_TAGS") or "").strip()
WANDB_LOG_ARTIFACT = (os.environ.get("WANDB_LOG_ARTIFACT") or "").strip()
WANDB_DISABLED = (os.environ.get("WANDB_DISABLED") or "").strip()
# Uncomment to group runs in the UI or upload the CSV artifact (can be large):
# WANDB_RUN_GROUP = "colab_" + __import__("time").strftime("%Y%m%d_%H%M")
# WANDB_LOG_ARTIFACT = "1"


def _is_wandb_env_line(line: str) -> bool:
    s = line.strip()
    if not s or s.startswith("#") or "=" not in s:
        return False
    return s.split("=", 1)[0].strip().startswith("WANDB_")


env_path = f"{SKILLS}/.env"
prev = open(env_path, encoding="utf-8").read() if os.path.isfile(env_path) else ""
kept = [ln for ln in prev.splitlines() if not _is_wandb_env_line(ln)]

wb = [
    "",
    "# Weights & Biases (optional — eval_runner logs when WANDB_PROJECT is set)",
    f"WANDB_PROJECT={WANDB_PROJECT}",
]
if WANDB_API_KEY:
    wb.append(f"WANDB_API_KEY={WANDB_API_KEY}")
if WANDB_ENTITY:
    wb.append(f"WANDB_ENTITY={WANDB_ENTITY}")
if WANDB_RUN_GROUP:
    wb.append(f"WANDB_RUN_GROUP={WANDB_RUN_GROUP}")
if WANDB_RUN_NAME:
    wb.append(f"WANDB_RUN_NAME={WANDB_RUN_NAME}")
if WANDB_TAGS:
    wb.append(f"WANDB_TAGS={WANDB_TAGS}")
if WANDB_LOG_ARTIFACT:
    wb.append(f"WANDB_LOG_ARTIFACT={WANDB_LOG_ARTIFACT}")
if WANDB_DISABLED:
    wb.append(f"WANDB_DISABLED={WANDB_DISABLED}")

out = "\n".join(kept + wb).rstrip() + "\n"
open(env_path, "w", encoding="utf-8").write(out)

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
else:
    os.environ.pop("WANDB_API_KEY", None)
for k, v in (
    ("WANDB_ENTITY", WANDB_ENTITY),
    ("WANDB_RUN_GROUP", WANDB_RUN_GROUP),
    ("WANDB_RUN_NAME", WANDB_RUN_NAME),
    ("WANDB_TAGS", WANDB_TAGS),
    ("WANDB_LOG_ARTIFACT", WANDB_LOG_ARTIFACT),
    ("WANDB_DISABLED", WANDB_DISABLED),
):
    if v:
        os.environ[k] = v
    else:
        os.environ.pop(k, None)

print("merged wandb into", env_path, "(secrets not echoed)")
print(
    f"WANDB_PROJECT={WANDB_PROJECT!r}, api_key_set={bool(WANDB_API_KEY)}, "
    f"entity_set={bool(WANDB_ENTITY)}, run_group_set={bool(WANDB_RUN_GROUP)}"
)


merged wandb into /content/work/SkillsAgent/.env (secrets not echoed)
WANDB_PROJECT='skillsagent-colab', api_key_set=True, entity_set=False, run_group_set=False


## 5. Smoke-test real TSFM

First run compiles the model; subsequent calls are <2 s. A successful run prints `forecast source: tsfm_subprocess` and `tsad_records` > 0.

In [7]:
import sys, time
sys.path.insert(0, SKILLS)
os.chdir(SKILLS)

for m in list(sys.modules):
    if m in ("tools", "knowledge", "agent", "skills", "confidence_evaluator"):
        del sys.modules[m]

from dotenv import load_dotenv
load_dotenv(f"{SKILLS}/.env", override=True)
from tools import get_sensor_data, forecast_sensor, deep_tsfm_refine_anomalies, detect_anomaly

d = get_sensor_data("Chiller 6", lookback_days=14)
print("IoT source:", d["source"], "rows:", d.get("iot_total_observations"))

t0 = time.time()
f = forecast_sensor("Chiller 6", "flow_rate_GPM", horizon_days=7, sensor_data=d)
print(f"forecast source={f['source']}  elapsed={time.time()-t0:.1f}s  head={f.get('forecasted', [])[:3]}")

basic = detect_anomaly(d)
t0 = time.time()
deep = deep_tsfm_refine_anomalies("Chiller 6", d, basic)
print(
    f"deep tsad  refined={deep.get('deep_tsfm_refined')}"
    f"  tsad_records={deep.get('tsfm_integrated_tsad_records', 0)}"
    f"  anomalies_detected={deep.get('anomalies_detected')}"
    f"  severity={deep.get('severity')}"
    f"  n_details={len(deep.get('anomaly_details', []))}"
    f"  elapsed={time.time()-t0:.1f}s"
)

IoT source: iot_csv rows: 1500
forecast source=tsfm_subprocess  elapsed=14.3s  head=[0.6911484003067017, 0.5683184862136841, 0.6224347352981567]
deep tsad  refined=True  tsad_records=45  anomalies_detected=True  severity=high  n_details=15  elapsed=15.3s


## 6a. Calibrate per-skill wall-clock latency on this GPU

Runs each skill 3× on a representative task and writes `skill_costs.json`. The evaluator picks this up automatically so Condition~E's `cost_budget` reflects T4 latencies, not Mac-CPU priors.

In [8]:
!cd {SKILLS} && python scripts/calibrate_costs.py --runs 3 --output skill_costs.json

import json
print(json.dumps(json.load(open(f'{SKILLS}/skill_costs.json')), indent=2))

Calibrating 7 skills × 3 runs (task: 'Diagnose anomalies on Chiller 6 and generate a work order')

  data_retrieval           median=  0.010s   runs=['0.280', '0.010', '0.010']
  metadata_retrieval       median=  2.346s   runs=['3.685', '2.346', '2.283']
  anomaly_detection        median=  6.021s   runs=['6.128', '6.021', '5.922']
  root_cause_analysis      median= 22.562s   runs=['22.562', '22.468', '22.905']
  validate_failure         median=  0.000s   runs=['0.000', '0.000', '0.000']
  forecasting              median= 18.753s   runs=['18.753', '15.586', '19.050']
  work_order_generation    median=  0.882s   runs=['0.891', '0.871', '0.882']

  __deep_tsfm__            median= 15.286s

wrote skill_costs.json  (50.57s total per full plan)
{
  "data_retrieval": 0.0102,
  "metadata_retrieval": 2.3463,
  "anomaly_detection": 6.0205,
  "root_cause_analysis": 22.5621,
  "validate_failure": 0.0,
  "forecasting": 18.7527,
  "work_order_generation": 0.8816,
  "__deep_tsfm__": 15.2862
}


## 6. Run eval on the representative subset

Without `--tsfm-report` or `--scenario_file` provided, uses the built-in `BUILTIN_TASK_BANK` (no HF download). To sweep more scenarios pass `--hf-limit 20`. All conditions A–F run; theta sweep included.

In [ ]:
OUT = f"{SKILLS}/eval_results/colab_{time.strftime('%Y%m%d_%H%M')}"
os.makedirs(OUT, exist_ok=True)

!cd {SKILLS} && python -m eval_runner \
    --output-dir {OUT} \
    --tsfm-report {TSFM_REPORT} \
    --trajectory-log {OUT}/trajectories.jsonl 2>&1

print("\nfiles:")
!ls -la {OUT}

In [10]:
OUT = f"{SKILLS}/eval_results/colab_{time.strftime('%Y%m%d_%H%M')}"
os.makedirs(OUT, exist_ok=True)

!cd {SKILLS} && python -m eval_runner \
    --output-dir {OUT} \
    --scenario-file {SKILLS}/eval_inputs/tsfm_set/scenarios.jsonl \
    --trajectory-log {OUT}/trajectories.jsonl 2>&1

print("\nfiles:")
!ls -la {OUT}

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: vmm2146 (vmm2146-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: ⢿ setting up run txmfaebu (0.0s)
wandb: ⣻ setting up run txmfaebu (0.0s)
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/work/SkillsAgent/wandb/run-20260503_234910-txmfaebu
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run dandy-galaxy-1
wandb: ⭐️ View project at https://wandb.ai/vmm2146-columbia-university/skillsagent-colab
wandb: 🚀 View run at https://wandb.ai/vmm2146-columbia-university/skillsagent-colab/runs/txmfaebu

────────────────────────────────────────────────────────────
Condition: A_raw_llm
────────────────────────────────────────────────────────────
  201 [fault_diagnosi

## 7. Copy results back to Drive

So you keep them after the runtime is recycled.

In [11]:
DRIVE_OUT = f"{DRIVE_PROJECT}/SkillsAgent/eval_results/{os.path.basename(OUT)}"
os.makedirs(DRIVE_OUT, exist_ok=True)
subprocess.check_call(["rsync", "-a", f"{OUT}/", f"{DRIVE_OUT}/"])
print("copied →", DRIVE_OUT)

copied → /content/drive/MyDrive/HPML_Project/SkillsAgent/eval_results/colab_20260503_2349
